# Classification trees: questions, partitions, and clinical decisions

**Lecture 14 · Notebook 00 · CMOR 438 / INDE 577**

A classification tree turns a sequence of yes/no questions into a probability
estimate and a class decision. We will derive the split criterion, implement and
inspect the course tree API, and evaluate a breast-mass classifier without
confusing a readable path with a medical or causal explanation.


## How to use this notebook

Run the **core** cells in order with the **Rice DSM** kernel. Pause at every
checkpoint and predict the result before executing it. **Practice** cells are
partially scaffolded; **extension** work belongs in a project branch. The
notebook is the experimental narrative. Durable algorithms belong in `src/`,
behavioral contracts in `tests/`, and reusable data access in package code.

If the kernel is missing, run `uv sync --locked` and
`uv run python scripts/setup_course.py` from the repository root.


## Learning objectives

By the end, you should be able to:

1. interpret a tree as recursive axis-aligned partitioning;
2. compute Gini impurity, entropy, and weighted impurity reduction;
3. explain greedy split search, stopping rules, and cost-complexity pruning;
4. preserve training, validation, and test boundaries;
5. evaluate probability and class decisions against a declared baseline; and
6. package, test, and document a tree as a professional software artifact.


## Why this matters

Trees are useful because they learn nonlinear thresholds and interactions without
feature scaling. Their history also teaches an important engineering lesson:
readable structure does not guarantee stable or causal structure. Morgan and
Sonquist's automatic interaction detection, Breiman and colleagues' CART, and
Quinlan's ID3/C4.5 family established related recursive-partitioning traditions.
Modern libraries optimize these ideas, while ensembles deliberately address the
high variance of a single tree.

**Professional practice:** begin with a measurable decision and a baseline. A
clinical false negative and false positive have different consequences; accuracy
alone cannot specify that tradeoff. This notebook is instructional, not a medical
device or clinical recommendation.


In [ ]:
# Standard-library imports handle paths, the local SQLite connection, and diagnostics.
from pathlib import Path
import sqlite3
import sys

# Scientific libraries support tables, numerical work, and visual evidence.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Import the maintained course implementation instead of redefining it in a cell.
from rice_dsm.ml.trees import (
    DecisionTreeClassifier,
    entropy_impurity,
    gini_impurity,
    plot_tree,
)
from rice_dsm.paths import course_database_path, find_project_root

# Resolve project resources independently of macOS, Linux, or Windows path syntax.
PROJECT_ROOT = find_project_root(Path.cwd())
DATABASE_PATH = course_database_path(PROJECT_ROOT)
print("Python:", sys.executable)
print("Database:", DATABASE_PATH)
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve()
assert DATABASE_PATH.is_file()


## 1. Start with the scientific question and inspect the data contract

The Wisconsin Diagnostic Breast Cancer observations summarize geometric features
computed from digitized images of cell nuclei. The target encoding in this course
database is `0 = malignant`, `1 = benign`. That ordering matters: a metric that
silently treats `1` as the clinically urgent event would tell the wrong story.

Before modeling, verify the observational unit, schema, missingness, duplicates,
target balance, and plausible ranges. Getting data is part of the work: here we
issue a read-only SQL query rather than asking a library to materialize a hidden
dataset. See Python's [`sqlite3` documentation](https://docs.python.org/3/library/sqlite3.html).


In [ ]:
# Open SQLite in read-only mode so exploration cannot silently alter course data.
database_uri = f"file:{DATABASE_PATH.as_posix()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    cancer = pd.read_sql_query(
        "SELECT * FROM breast_cancer_observations ORDER BY observation_id",
        connection,
    )

# Keep one explicit ordered feature contract for fitting, prediction, and plots.
target_name = "diagnosis_code"
feature_names = [
    "mean_radius",
    "mean_texture",
    "mean_concavity",
    "worst_radius",
    "worst_texture",
    "worst_concave_points",
]
# Inspect rows, numerical summaries, and class support before choosing a model.
display(cancer.head())
print(cancer[[*feature_names, target_name]].describe().T)
print("Class counts:\n", cancer["diagnosis_label"].value_counts())

# Executable assertions turn our data assumptions into immediate failure messages.
assert cancer.shape == (569, 33)
assert cancer[feature_names + [target_name]].isna().sum().sum() == 0
assert cancer["observation_id"].is_unique
assert set(cancer[target_name]) == {0, 1}


In [ ]:
# Visual evidence asks whether two measurements show thresholds or interactions.
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(
    data=cancer,
    x="worst_radius",
    y="worst_concave_points",
    hue="diagnosis_label",
    alpha=0.72,
    ax=axes[0],
)
# Summarize one marginal distribution without pretending overlap disappears.
sns.boxplot(
    data=cancer,
    x="diagnosis_label",
    y="worst_radius",
    hue="diagnosis_label",
    legend=False,
    ax=axes[1],
)
axes[0].set_title("Two measurements do not separate every observation")
axes[1].set_title("A useful feature still has overlapping classes")
figure.tight_layout()


## 2. The mathematical object: a recursive partition

At an internal node containing index set $S$, a candidate question is
$x_j \le t$. It forms children $S_L$ and $S_R$. If $p_k$ is the fraction of
node observations in class $k$, then

$$G(S)=1-\sum_k p_k^2, \qquad
  H(S)=-\sum_k p_k\log_2 p_k.$$

These formulas answer: **how unresolved are the labels before asking another
question?** Gini impurity is the probability that two independent labels drawn
from the node distribution disagree. Entropy is the average surprise, measured
in bits, of learning a label drawn from that distribution. Both are zero when the
node is pure and largest when the classes are evenly represented.

A third quantity, classification error $1-\max_k p_k$, is the error made by
predicting the majority class. It is intuitive but less sensitive to changes in
class proportions, so Gini and entropy provide a more informative search signal.
None of the three is the downstream cost of a false positive or false negative.

A split is scored by weighted impurity reduction

$$\Delta I = I(S)-\frac{|S_L|}{|S|}I(S_L)-\frac{|S_R|}{|S|}I(S_R).$$

Weighting is essential: it computes the impurity expected for a randomly chosen
observation *after* the question. Without the weights, isolating one pure example
could look as valuable as organizing hundreds of observations. The reduction is
therefore the immediate uncertainty removed by the split. The algorithm greedily
chooses the largest immediate reduction; this is not proof that the globally
optimal tree has been found.


In [ ]:
# Compare the three node measures for every binary class proportion.
probability = np.linspace(0.001, 0.999, 500)
gini_curve = 2 * probability * (1 - probability)
entropy_curve = -(
    probability * np.log2(probability)
    + (1 - probability) * np.log2(1 - probability)
)
classification_error = 1 - np.maximum(probability, 1 - probability)

# Compare curve shape on one axis; each vertical scale retains its own meaning.
figure, axis = plt.subplots(figsize=(9, 4.8))
axis.plot(probability, gini_curve, label="Gini: probability of disagreement")
axis.plot(probability, entropy_curve, label="entropy: average surprise in bits")
axis.plot(
    probability,
    classification_error,
    linestyle="--",
    label="majority-class error",
)
axis.axvline(0.5, color="black", linewidth=1, alpha=0.4)
axis.set(
    xlabel="proportion in class 1",
    ylabel="node uncertainty / error",
    title="Pure nodes score zero; evenly mixed nodes are most uncertain",
)
axis.legend(frameon=False)
figure.tight_layout()


### Worked example: calculate one split by hand

Suppose the parent has four malignant and four benign observations. A proposed
split produces left counts $(4,1)$ and right counts $(0,3)$. Compute the parent,
child, and weighted impurities before running the cell.


In [ ]:
# Use class counts, not raw labels, because impurity depends only on proportions.
parent_counts = np.array([4, 4])
left_counts = np.array([4, 1])
right_counts = np.array([0, 3])

# Weight each child by the probability that an observation reaches that child.
parent_gini = gini_impurity(parent_counts)
weighted_children = (
    left_counts.sum() * gini_impurity(left_counts)
    + right_counts.sum() * gini_impurity(right_counts)
) / parent_counts.sum()
gini_gain = parent_gini - weighted_children

# The gain of 0.3 is the expected disagreement removed by asking this question.
print(f"parent Gini = {parent_gini:.3f}")
print(f"weighted child Gini = {weighted_children:.3f}")
print(f"impurity reduction = {gini_gain:.3f}")
print(f"parent entropy = {entropy_impurity(parent_counts):.3f} bits")

assert parent_gini == 0.5
assert np.isclose(weighted_children, 0.2)
assert np.isclose(gini_gain, 0.3)


In [ ]:
# Enumerate candidate thresholds for one feature to expose greedy split search.
toy_x = np.array([1.0, 2.0, 3.0, 5.0, 8.0, 9.0])
toy_y = np.array([0, 0, 1, 0, 1, 1])
candidates = (np.unique(toy_x)[:-1] + np.unique(toy_x)[1:]) / 2
# Retain every score so an exact tie cannot be hidden by argmin.
rows = []
for threshold in candidates:
    left = toy_y[toy_x <= threshold]
    right = toy_y[toy_x > threshold]
    left_counts = np.bincount(left, minlength=2)
    right_counts = np.bincount(right, minlength=2)
    weighted = (
        len(left) * gini_impurity(left_counts)
        + len(right) * gini_impurity(right_counts)
    ) / len(toy_y)
    rows.append({"threshold": threshold, "weighted_gini": weighted})

candidate_table = pd.DataFrame(rows)
display(candidate_table.style.highlight_min(subset="weighted_gini", color="#d9ead3"))
best_score = candidate_table["weighted_gini"].min()
tied_thresholds = candidate_table.loc[
    np.isclose(candidate_table["weighted_gini"], best_score), "threshold"
].tolist()
print("Equally good thresholds:", tied_thresholds)
assert tied_thresholds == [2.5, 6.5]


## 3. Freeze the evaluation boundary before choosing a tree

We use three roles:

- **training data** estimates thresholds and leaf probabilities;
- **validation data** chooses depth and leaf-size constraints;
- **test data** estimates performance once after the choice is frozen.

Stratification preserves class proportions. Unlike distance-based models, a tree
does not require standardization: strictly monotone transformations preserve the
ordering of candidate thresholds. Preprocessing can still leak information, and
missing-value or category policies must still be learned inside the training
boundary.


In [ ]:
# Create the untouched test set first; it is used only after model selection.
from sklearn.model_selection import train_test_split

# Convert named table columns into the exact ordered matrix contract used by fit.
X = cancer[feature_names].to_numpy(dtype=float)
y = cancer[target_name].to_numpy(dtype=int)
# Stratification preserves class proportions in each evaluation partition.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=438
)
# Split the remaining data again: training fits; validation selects structure.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=438,
)

assert len(X_train) + len(X_validation) + len(X_test) == len(X)
assert set(map(tuple, X_train)).isdisjoint(set(map(tuple, X_test)))
print(len(X_train), len(X_validation), len(X_test))


### Worked example: fit an inspectable shallow tree

A depth-two model is small enough to audit. Its exported rules are a faithful
description of this fitted artifact. They are not biological mechanisms, causal
claims, or guaranteed-stable population facts.


In [ ]:
# A depth-two tree is small enough to audit every question and leaf by hand.
course_tree = DecisionTreeClassifier(
    criterion="gini", max_depth=2, min_samples_leaf=12
).fit(X_train, y_train)

# Text rules are useful for tests and logs; the next cell shows the hierarchy.
print(course_tree.export_text(feature_names))
print("depth:", course_tree.get_depth(), "leaves:", course_tree.get_n_leaves())
assert course_tree.get_depth() <= 2
assert np.isclose(course_tree.feature_importances_.sum(), 1.0)
assert course_tree.predict_proba(X_validation[:5]).shape == (5, 2)


In [ ]:
# Visualize the same fitted course tree--not a separately fitted illustration.
figure, axis = plt.subplots(figsize=(12, 6))
plot_tree(course_tree, feature_names, ax=axis, precision=2)
axis.set_title(
    "Classification tree: questions lead to empirical class distributions",
    color="#00205b",
    weight="bold",
)
figure.tight_layout()


In [ ]:
# Compare declared candidates on validation data; do not touch test labels here.
from sklearn.metrics import balanced_accuracy_score

# Search a small declared grid so the model-selection process remains auditable.
candidates = []
for criterion in ("gini", "entropy"):
    for max_depth in (1, 2, 3, 4, 6):
        for min_leaf in (5, 12, 24):
            model = DecisionTreeClassifier(
                criterion=criterion,
                max_depth=max_depth,
                min_samples_leaf=min_leaf,
            ).fit(X_train, y_train)
            score = balanced_accuracy_score(y_validation, model.predict(X_validation))
            candidates.append(
                {
                    "criterion": criterion,
                    "max_depth": max_depth,
                    "min_leaf": min_leaf,
                    "validation_balanced_accuracy": score,
                    "leaves": model.get_n_leaves(),
                }
            )

validation_results = pd.DataFrame(candidates).sort_values(
    ["validation_balanced_accuracy", "leaves"], ascending=[False, True]
)
best = validation_results.iloc[0]
display(validation_results.head(10))
assert 0.0 <= best["validation_balanced_accuracy"] <= 1.0


## 4. Refit, compare with a baseline, and evaluate once

After selecting hyperparameters, refit on training plus validation data. Report a
dummy baseline, balanced accuracy, class-specific recall, the confusion matrix,
and probability quality. Here malignant is encoded as class 0, so we state the
class ordering rather than relying on metric defaults.


In [ ]:
# Freeze the chosen tree, compare a baseline, and report class-aware evidence.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    log_loss,
)

final_tree = DecisionTreeClassifier(
    criterion=str(best["criterion"]),
    max_depth=int(best["max_depth"]),
    min_samples_leaf=int(best["min_leaf"]),
).fit(X_train_full, y_train_full)
baseline = DummyClassifier(strategy="prior").fit(X_train_full, y_train_full)

# Hard decisions and probability vectors support different evaluation questions.
predictions = final_tree.predict(X_test).astype(int)
probabilities = final_tree.predict_proba(X_test)
baseline_predictions = baseline.predict(X_test)
print("baseline balanced accuracy:", balanced_accuracy_score(y_test, baseline_predictions))
print("tree balanced accuracy:", balanced_accuracy_score(y_test, predictions))
print("test log loss:", log_loss(y_test, probabilities, labels=final_tree.classes_))
print(classification_report(y_test, predictions, target_names=["malignant", "benign"]))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["malignant", "benign"],
    cmap="Blues",
)
plt.title("Held-out errors must be read with class consequences")
plt.show()

assert len(predictions) == len(y_test)
assert np.allclose(probabilities.sum(axis=1), 1.0)


## 5. Pruning is regularization, not decoration

A fully grown tree can isolate tiny leaves and drive training impurity toward zero.
Cost-complexity pruning balances fit and terminal-node count:

$$R_\alpha(T)=R(T)+\alpha |\widetilde T|.$$

The pruning path is estimated on training data. The value of $\alpha$ is selected
with validation evidence, never by looking for the best test result. Structural
constraints such as `max_depth` and `min_samples_leaf` are also regularizers.


In [ ]:
# Build the weakest-link pruning path using the trusted optimized implementation.
from sklearn.tree import DecisionTreeClassifier as SklearnTreeClassifier

unpruned = SklearnTreeClassifier(random_state=438, min_samples_leaf=3)
path = unpruned.cost_complexity_pruning_path(X_train, y_train)
alpha_grid = np.unique(path.ccp_alphas)
# Score each candidate subtree on validation data, never on final test labels.
pruning_rows = []
for alpha in alpha_grid:
    model = SklearnTreeClassifier(
        random_state=438, min_samples_leaf=3, ccp_alpha=float(alpha)
    ).fit(X_train, y_train)
    pruning_rows.append(
        {
            "alpha": alpha,
            "depth": model.get_depth(),
            "leaves": model.get_n_leaves(),
            "validation_score": balanced_accuracy_score(
                y_validation, model.predict(X_validation)
            ),
        }
    )

pruning = pd.DataFrame(pruning_rows)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].step(pruning["alpha"], pruning["leaves"], where="post")
axes[0].set(xlabel="cost-complexity alpha", ylabel="leaf count", title="Penalty simplifies structure")
axes[1].plot(pruning["alpha"], pruning["validation_score"], marker="o")
axes[1].set(xlabel="cost-complexity alpha", ylabel="balanced accuracy", title="Validation selects complexity")
figure.tight_layout()
assert pruning.iloc[-1]["leaves"] == 1


## 6. Stability and interpretation have a scope

Small sample changes can alter an early split and every descendant. We therefore
examine root-feature frequency across bootstrap refits. Impurity importance is a
model-specific allocation of training impurity reduction; it can favor variables
with many candidate splits and divide credit unpredictably among correlated
variables. Permutation importance asks a different predictive question and can
also become unrealistic when features are dependent.


In [ ]:
# Refit shallow trees on bootstrap samples and record the first question.
rng = np.random.default_rng(438)
root_features = []
for _ in range(80):
    sample = rng.integers(0, len(X_train_full), size=len(X_train_full))
    boot_tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=12).fit(
        X_train_full[sample], y_train_full[sample]
    )
    root_features.append(feature_names[boot_tree.tree_.feature_index])

root_frequency = pd.Series(root_features).value_counts(normalize=True)
display(root_frequency.rename("bootstrap root frequency"))
root_frequency.sort_values().plot.barh(title="A readable root may not be stable")
plt.xlabel("fraction of bootstrap fits")
plt.show()
assert np.isclose(root_frequency.sum(), 1.0)


In [ ]:
# Boundary tests encode the exact <= routing contract around the fitted root.
root = final_tree.tree_
assert root.feature_index is not None and root.threshold is not None
reference = X_test[0].copy()
below = reference.copy()
equal = reference.copy()
above = reference.copy()
below[root.feature_index] = np.nextafter(root.threshold, -np.inf)
equal[root.feature_index] = root.threshold
above[root.feature_index] = np.nextafter(root.threshold, np.inf)

print("root feature:", feature_names[root.feature_index])
print("root threshold:", root.threshold)
print("leaf ids below/equal/above:", final_tree.apply([below, equal, above]))
assert final_tree.apply([below])[0] == final_tree.apply([equal])[0]


## 7. Professional engineering: move the algorithm out of the notebook

The course implementation lives in `src/rice_dsm/ml/trees.py`; analytic and
failure contracts live in `tests/test_decision_trees.py`. Create or update your
project from the [companion student template](https://github.com/RandyRDavila/Data-Science-Machine-Learning-Project-Template).
From a clean template `main`, run:

```text
git switch main
git pull --ff-only
git switch -c YOUR-GITHUB-USERNAME/classification-tree
uv sync --locked
```

Reproduce this boundary:

- `src/dsm_project/algorithms/student/trees.py`: typed estimator API;
- `src/dsm_project/algorithms/registry.py`: explicit estimator registration;
- `tests/test_student_trees.py`: hand calculations, threshold equality, invalid
  inputs, fit-before-predict, and a trusted-library differential test;
- `data/local/`: student-owned raw data excluded from Git when licensing or size
  requires it;
- `data/fixtures/`: a tiny redistributable CI fixture;
- `notebooks/`: the question-led applied analysis importing the package.

Work in this order: open an issue defining the prediction question and data
contract; write the analytic tests; implement one small behavior at a time; run
the tests; then write the applied notebook that imports your package. Do not paste
the estimator class into the notebook.

Before pushing, run:

```text
uv run ruff check src tests scripts
uv run pytest -q
git status --short
git add src tests notebooks docs data/fixtures
git commit -m "Add classification-tree case study"
git push -u origin YOUR-GITHUB-USERNAME/classification-tree
```

Your pull request should link the issue and explain the question, dataset source
and license, split policy, baseline, metrics, limitations, tests, and one failure
you investigated. Never commit private data merely because it fits in Git.


### Student contribution test plan

| Test | Contract protected |
| --- | --- |
| counts $(2,2)$ give Gini $0.5$ and entropy $1$ | impurity mathematics |
| four-row fixture selects threshold $1.5$ | greedy split correctness |
| a value equal to the threshold routes left | boundary semantics |
| leaf probabilities sum to one in class order | probability API |
| invalid shape, NaN, and pre-fit prediction fail clearly | user-facing failures |
| fixed configuration agrees with scikit-learn | differential behavior |
| clean-kernel notebook execution | package/notebook boundary |

Tests protect public behavior, not private helper names. A useful failure tells a
teammate which mathematical or interface contract broke.


In [ ]:
# This smoke test demonstrates the import boundary students should reproduce.
# In the template repository, replace the next import with:
# from dsm_project.algorithms.student.trees import DecisionTreeClassifier
from rice_dsm.ml.trees import DecisionTreeClassifier as CourseClassifier

smoke_classifier = CourseClassifier(max_depth=1).fit(
    [[0.0], [1.0], [2.0], [3.0]],
    ["low", "low", "high", "high"],
)
assert smoke_classifier.predict([[0.25], [2.75]]).tolist() == ["low", "high"]
print("The notebook imported tested package code; it did not redefine the tree.")


## Professional practice and common failure modes

- **Leakage:** selecting depth or pruning strength on test labels.
- **Metric blindness:** reporting accuracy without class support or consequences.
- **Probability overconfidence:** treating small-leaf frequencies as calibrated.
- **Schema drift:** changing feature order, units, or category encodings.
- **Interpretation overreach:** calling a fitted path causal or biologically true.
- **Instability:** publishing one tree diagram without resampling evidence.
- **Missing values:** allowing an implicit library default instead of a declared
  reject/impute/native-missingness policy.


## Debugging clinic

If predictions disagree with a hand calculation, inspect in this order:

1. target class ordering (`model.classes_`);
2. candidate threshold midpoints and the `<=` equality rule;
3. weighted rather than unweighted child impurity;
4. feature column order at fit and predict time;
5. whether preprocessing was fit using validation or test data; and
6. leaf counts before class labels or probabilities are returned.

A minimal four-row fixture is more useful for debugging split semantics than the
full clinical dataset.


## Guided practice

Compare a one-level decision stump using Gini and entropy. Explain why their
numerical impurity values differ and whether the selected root question differs.
Then add a unit test for your claim before changing implementation code.


In [ ]:
# PRACTICE: fit two stumps and compare their selected root questions.
gini_stump = DecisionTreeClassifier(criterion="gini", max_depth=1).fit(X_train, y_train)
entropy_stump = DecisionTreeClassifier(criterion="entropy", max_depth=1).fit(X_train, y_train)

print(gini_stump.export_text(feature_names))
print(entropy_stump.export_text(feature_names))
assert gini_stump.get_depth() == 1
assert entropy_stump.get_depth() == 1
# TODO: write one sentence interpreting agreement or disagreement.


## Independent practice

Add a classification-tree analysis to your project using your chosen dataset.
Your notebook must state the observational unit and decision, query or load data
through a documented boundary, compare against a dummy baseline, select complexity
without test leakage, include class-conditional errors, and import your package.
Open an issue first and link the issue from the pull request.


## Extension: probabilities, calibration, and asymmetric decisions

Use repeated cross-validation to compare leaf probabilities with calibrated
probabilities, then choose a decision threshold under an explicit false-negative
cost. Do not claim clinical utility: document what prospective evidence would be
required. A stronger extension compares stability of predictions and paths.


## Retrieval practice

1. Why must child impurity be weighted by child sample counts?
2. Why can a leaf be pure on training data and unreliable in deployment?
3. What does `min_samples_leaf` regularize?
4. Why does a readable path not establish causality?
5. Which exact boundary case should a threshold unit test include?


## Takeaway and further reading

A classification tree is a greedy, piecewise-constant probability estimator. Its
criteria, tie policy, stopping rules, pruning, class ordering, and threshold
semantics are part of the model contract. Evaluate the decision with held-out
evidence, and interpret paths only as descriptions of one fitted artifact.

Further reading: [scikit-learn decision-tree user guide](https://scikit-learn.org/stable/modules/tree.html),
[Python data model](https://docs.python.org/3/reference/datamodel.html), and the
original CART monograph by Breiman, Friedman, Olshen, and Stone (1984).
